In [5]:
from dotenv import load_dotenv
import os
load_dotenv()

API_KEY=os.getenv("GOOGLE_API_KEY")

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash",api_key=API_KEY)


SystemMessage </br>
HumanMessage </br>
AIMessage </br>



In [8]:
from langchain_core.messages import SystemMessage,HumanMessage


messages=[
    SystemMessage(content="You are a engineer a some FAAG company have strong knowledege of CS fundamentals"),
    HumanMessage(content="Teach me about DNS")
]

result=llm.invoke(messages)

In [9]:
result

AIMessage(content="Alright, let's dive into the fascinating world of DNS (Domain Name System). Think of DNS as the internet's phonebook.  It translates human-friendly domain names (like `google.com`) into machine-readable IP addresses (like `142.250.185.142`), which computers use to locate each other on the network.\n\nHere's a breakdown covering the key concepts:\n\n**1. Why DNS is Necessary**\n\n*   **Humans vs. Machines:**  We humans are good at remembering names, but computers work best with numerical addresses.  Imagine having to remember the IP address for every website you visit! DNS makes the internet much more user-friendly.\n*   **IP Address Changes:** IP addresses can change.  If a website's IP address changes, DNS allows the domain name to be updated to point to the new IP address without requiring users to change anything on their end.\n*   **Scalability:**  DNS is a distributed and hierarchical system, which allows it to scale to handle the vast number of websites and dev

Chunk Response


In [ ]:
for chunk in llm.stream("tell me about DNS"):
    print(chunk.content, end="", flush=True)


Template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate


template ="write a {tone} email to {company} expressing interest in the {position} ,mentioning {skill} as the key strength. keep is to 4 line max"
prompt_template=ChatPromptTemplate.from_template(template=template)
prompt=prompt_template.invoke({
    "tone":"enegetic",
    "company":"samsung",
    "position":"AI engineer",
    "skill":"AI"
})


print(prompt)

messages=[HumanMessage(content='write a enegetic email to samsung expressing interest in the AI engineer ,mentioning AI as the key strength. keep is to 4 line max', additional_kwargs={}, response_metadata={})]


Better way


In [19]:
messages=[
    ("system","somtoiasfsa sf {v1} asfa"),
    ("human","something someasf asf {v2}")
]

prompt_template=ChatPromptTemplate.from_messages(messages)
prompt=prompt_template.invoke({
    "v1":"safasf",
    "v2":"sadasd"
})

print(prompt_template,prompt)


input_variables=['v1', 'v2'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['v1'], input_types={}, partial_variables={}, template='somtoiasfsa sf {v1} asfa'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['v2'], input_types={}, partial_variables={}, template='something someasf asf {v2}'), additional_kwargs={})] messages=[SystemMessage(content='somtoiasfsa sf safasf asfa', additional_kwargs={}, response_metadata={}), HumanMessage(content='something someasf asf sadasd', additional_kwargs={}, response_metadata={})]


## Chaining
- sequential chaining
- parallel chaining
- conditional chaining

# har ka output next step me insert hoga


# sequential chaining


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
messages=[
    ("system","you are {animal} specialist"),
    ("human","Tell me {number} facts about {animal}")
]

prompt_template=ChatPromptTemplate.from_messages(messages)


chain=prompt_template|llm|StrOutputParser()
for i in chain:
    print(i)
result=chain.invoke({"animal":"dog","number":"2"})

('name', None)
('first', ChatPromptTemplate(input_variables=['animal', 'number'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['animal'], input_types={}, partial_variables={}, template='you are {animal} specialist'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['animal', 'number'], input_types={}, partial_variables={}, template='Tell me {number} facts about {animal}'), additional_kwargs={})]))
('middle', [ChatGoogleGenerativeAI(model='models/gemini-2.0-flash', google_api_key=SecretStr('**********'), client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x000001C3A5171C50>, default_metadata=()), ChatGoogleGenerativeAI(model='models/gemini-2.0-flash', google_api_key=SecretStr('**********'), client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x000001C3A5171C

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
messages=[
    ("system","you are {animal} specialist"),
    ("human","Tell me {number} facts about {animal}")
]

prompt_template=ChatPromptTemplate.from_messages(messages)
prompt=prompt_template.format_prompt(animal="dog",number="20")
print(prompt)

messages=[SystemMessage(content='you are dog specialist', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me 20 facts about dog', additional_kwargs={}, response_metadata={})]


In [ ]:
from langchain.schema.runnable import RunnableLambda
animal_facts_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You like telling facts and you tell facts about {animal}."),
        ("human", "Tell me {count} facts."),
    ]
)
translation_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a translator and convert the provided text into {language}."),
        ("human", "Translate the following text to {language}: {text}"),
    ]
)
prepare_for_translation = RunnableLambda(lambda output: {"text": output, "language": "french"})


chain = animal_facts_template | llm | StrOutputParser() | prepare_for_translation | translation_template | llm | StrOutputParser() 

# Run the chain
result = chain.invoke({"animal": "cat", "count": 2})

"Okay, here are two interesting facts about dogs:\n\n1.  **Dogs have a sense of time, and it affects their behavior:** Studies show that dogs can differentiate between durations of time. Dogs left alone for two hours show less excitement upon their owner's return than dogs left alone for five hours, indicating they perceive the difference in time passed.\n\n2.  **A dog's nose print is as unique as a human fingerprint:** Just like fingerprints, the pattern of ridges and creases on a dog's nose is unique to each individual dog. This could theoretically be used for identification purposes!"

# parallel chaining



In [ ]:
from langchain.schema.runnable import RunnableLambda,RunnableParallel

summary_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a movie critic."),
        ("human", "Provide a brief summary of the movie {movie_name}."),
    ]
)
def analyze_plot(plot):
    plot_template = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a movie critic."),
            ("human", "Analyze the plot: {plot}. What are its strengths and weaknesses?"),
        ]
    )
    return plot_template.format_prompt(plot=plot)

# Define character analysis step
def analyze_characters(characters):
    character_template = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a movie critic."),
            ("human", "Analyze the characters: {characters}. What are their strengths and weaknesses?"),
        ]
    )
    return character_template.format_prompt(characters=characters)

# Combine analyses into a final verdict
def combine_verdicts(plot_analysis, character_analysis):
    return f"Plot Analysis:\n{plot_analysis}\n\nCharacter Analysis:\n{character_analysis}"

# Simplify branches with LCEL
plot_branch_chain = (
    RunnableLambda(lambda x: analyze_plot(x)) | llm | StrOutputParser()
)

character_branch_chain = (
    RunnableLambda(lambda x: analyze_characters(x)) | llm | StrOutputParser()
)

In [22]:
analyze_plot("hmm")

ChatPromptValue(messages=[SystemMessage(content='You are a movie critic.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Analyze the plot: hmm. What are its strengths and weaknesses?', additional_kwargs={}, response_metadata={})])

In [ ]:
chain = (
    summary_template
    | llm
    | StrOutputParser()
    | RunnableParallel(branches={"plot": plot_branch_chain, "characters": character_branch_chain})
    | RunnableLambda(lambda x: combine_verdicts(x["branches"]["plot"], x["branches"]["characters"]))
)

# Run the chain
result = chain.invoke({"movie_name": "Inception"})

print(result)

# conditional chaining
- branches – A tuple of (condition, Runnable) pairs.
- default – A Runnable to run if no condition is met.

In [ ]:
from langchain.schema.runnable import RunnableBranch





positive_feedback_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human",
         "Generate a thank you note for this positive feedback: {feedback}."),
    ]
)

negative_feedback_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human",
         "Generate a response addressing this negative feedback: {feedback}."),
    ]
)

neutral_feedback_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        (
            "human",
            "Generate a request for more details for this neutral feedback: {feedback}.",
        ),
    ]
)

escalate_feedback_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        (
            "human",
            "Generate a message to escalate this feedback to a human agent: {feedback}.",
        ),
    ]
)







branches = RunnableBranch(
    (
        lambda x: "positive" in x , positive_feedback_template | llm | StrOutputParser()  # Positive feedback chain
        #condition                      runnable
    ),
    (
        lambda x: "negative" in x , negative_feedback_template | llm | StrOutputParser()  # Negative feedback chain
    ),
    (
        lambda x: "neutral" in x,
        neutral_feedback_template | llm | StrOutputParser()  # Neutral feedback chain
    ),
    
    escalate_feedback_template | llm | StrOutputParser()
    
    #default
)

classification_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human",
         "Classify the sentiment of this feedback as positive, negative, neutral, or escalate: {feedback}."),
    ]
)
classification_chain = classification_template | llm | StrOutputParser()

# Combine classification and response generation into one chain
chain = classification_chain | branches






review = "The product is terrible. It broke after just one use and the quality is very poor."
result = chain.invoke({"feedback": review})

# Output the result
print(result)





## Agent
- ReaACT = Reaction + Acting
- Think -> Act -> observe (repeat if required)


In [1]:
a=[("human","safasasf")]


In [2]:
a=[1,2,3,4,5]


In [4]:
a[:-1]+[1]

[1, 2, 3, 4, 1]